# 08 — Pulse Waveform Definition

## Purpose

Define and register all pulse waveforms needed for advanced experiments: constant (square) pulses for Fock-selective operations, DRAG-Gaussian pulses for the g→e and e→f qubit transitions, displacement operations for the storage cavity, and the associated rotation sets.

## Methodology

1. **Constant (square) pulse** — define amplitude and length for Fock-selective π and π/2 rotations
2. **DRAG-Gaussian (g→e)** — build derivative-reduction-by-adiabatic-gate (DRAG) corrected Gaussian pulses to minimize leakage to the |f⟩ state during g↔e drives
3. **DRAG-Gaussian (e→f)** — same waveform shape for the e↔f transition, using the anharmonicity to set the detuning
4. **Rotation registration** — derive the full rotation set (x180, x90, y180, y90, etc.) from each reference π-pulse via `register_rotations_from_ref_iq()`
5. **Displacement operations** — register coherent-state displacement pulses for the storage cavity via `ensure_displacement_ops()`
6. **Waveform validation** — plot all waveform I/Q envelopes for visual verification
7. **Calibration commit** — persist the pulse definitions via `preview_or_apply_patch_ops()`

## Expected Outcomes

- All pulse operations registered in the session's pulse manager
- I/Q envelopes smooth with amplitudes within [−1, 1] DAC range
- DRAG correction visible as an imaginary-component feature in the Gaussian envelope
- Displacement pulse properly scaled for the target cavity displacement amplitude

## Prerequisites

- **Notebook 05** — qubit frequency and π-pulse amplitude calibrated
- **Notebook 06** — coherence times measured (needed for pulse length optimization context)

## 1. Setup — Session Bootstrap

Open the notebook stage and load the prior coherence experiments checkpoint.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    drag_gaussian_pulse_waveforms,
    ensure_displacement_ops,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    register_rotations_from_ref_iq,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="08_pulse_waveform_definition",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

coherence_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="06_coherence_experiments",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"Closed a live in-memory session before reopen: {stage.had_live_session}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
print(f"Current qubit frequency: {float(attr.qb_fq) / 1e9:.6f} GHz")
print(f"Current anharmonicity: {float(attr.anharmonicity) / 1e6:.3f} MHz")
if coherence_checkpoint is not None:
    print(
        "Prior stage 06 status: "
        f"{coherence_checkpoint['status']}"
        f" ({coherence_checkpoint['summary']})"
    )

## 2. Configuration — Waveform Defaults

Define the pulse shape parameters for all waveform types. These values are typically refined iteratively: start with rough estimates from notebook 05, then adjust based on Rabi calibration results.

In [ ]:
APPLY_PULSE_WAVEFORM_CALIBRATION = True

# --- Constant (square) pulse defaults (Exp 8) ---
CONST_R180_AMP = 0.02488 / 2 * 0.8129
CONST_LEN = 200
CONST_R90_AMP = CONST_R180_AMP / 2.0

# --- DRAG-Gaussian g→e defaults (Exp 9) ---
GE_RLEN = 16
GE_SIGMA = GE_RLEN / 6
GE_R180_AMP = 0.12097039500000001 * 0.9815 * 1.031 * 0.9481 * 1.0606
GE_DRAG_COEFF = 0.3

# --- DRAG-Gaussian e→f defaults (Exp 10) ---
EF_RLEN = 16
EF_SIGMA = EF_RLEN / 6
EF_R180_AMP = 0.135 * 0.9847 * 0.91 * 1.0725
EF_DRAG_COEFF = 0.0019

# --- Per-gate fine tweaks (g→e) ---
GE_D_LAMBDA_MAP = {"x90": 0, "y90": 0, "xn90": 0, "yn90": 0, "x180": 0, "y180": 0}
GE_D_ALPHA_MAP = {"y90": 0, "yn90": 0, "x90": 0, "xn90": 0, "x180": 0, "y180": 0}
GE_D_OMEGA_MAP = {"y90": 0, "yn90": 0, "x90": 0, "xn90": 0, "x180": 0, "y180": 0}

# --- Per-gate fine tweaks (e→f) ---
EF_D_LAMBDA_MAP = {
    "x90": -2507242.9277225947, "y90": -2507242.9277225947,
    "xn90": -2507242.9277225947, "yn90": -2507242.9277225947,
    "x180": -1587694.276782522, "y180": -1587694.276782522,
}
EF_D_ALPHA_MAP = {
    "y90": 0.02182811504634108, "yn90": 0.02182811504634108,
    "x90": 0.02182811504634108, "xn90": 0.02182811504634108,
    "x180": -0.02306254983517153, "y180": -0.02306254983517153,
}
EF_D_OMEGA_MAP = {
    "y90": 62499, "yn90": 62499,
    "x90": 62499, "xn90": 62499,
    "x180": -62499.999999999985, "y180": -62499.999999999985,
}

print("Pulse waveform definition stage settings:")
print(f"  apply waveform calibration: {APPLY_PULSE_WAVEFORM_CALIBRATION}")
print(f"  const r180 amp: {CONST_R180_AMP:.6f}")
print(f"  const len: {CONST_LEN}")
print(f"  g→e DRAG: rlen={GE_RLEN}, sigma={GE_SIGMA:.3f}, amp={GE_R180_AMP:.6f}, drag={GE_DRAG_COEFF}")
print(f"  e→f DRAG: rlen={EF_RLEN}, sigma={EF_SIGMA:.3f}, amp={EF_R180_AMP:.6f}, drag={EF_DRAG_COEFF}")

## 3. Execution — Constant (Square) Pulse Waveforms

Build constant-amplitude I/Q waveforms and register the full rotation family with prefix `const_` (e.g., `const_x90`, `const_x180`). These fixed-amplitude pulses are used for Fock-selective operations where DRAG correction is not needed.

In [ ]:
def const_square_waveforms(amplitude_I, amplitude_Q, length):
    """Build a pair (I_wf, Q_wf) for a square (constant) pulse."""
    I_wf = [amplitude_I] * length
    Q_wf = [amplitude_Q] * length
    return I_wf, Q_wf

const_I_x180, const_Q_x180 = const_square_waveforms(CONST_R180_AMP, 0.0, CONST_LEN)

pom = session.pulse_mgr
const_reg_result = register_rotations_from_ref_iq(
    pom,
    const_I_x180,
    const_Q_x180,
    rotations="all",
    make_r0=True,
    prefix="const_",
    override=True,
    persist=True,
)

print(f"Registered constant pulse rotations: {list(const_reg_result.keys()) if isinstance(const_reg_result, dict) else const_reg_result}")
print(f"  const_r180 amp: {CONST_R180_AMP:.6f}")
print(f"  const_r90  amp: {CONST_R90_AMP:.6f}")
print(f"  length: {CONST_LEN} samples")

## 4. Execution — DRAG-Gaussian Waveforms (g→e Transition)

Build DRAG-corrected Gaussian waveforms for the ground → excited (g→e) transition at the qubit frequency. The DRAG derivative correction suppresses leakage to the |f⟩ state. Register the full rotation family (x90, y90, x180, y180, xn90, yn90, r0).

In [ ]:
ge_gauss_r180, ge_drag_r180 = drag_gaussian_pulse_waveforms(
    GE_R180_AMP, GE_RLEN, GE_SIGMA, GE_DRAG_COEFF, float(attr.anharmonicity)
)

ge_reg_result = register_rotations_from_ref_iq(
    pom,
    ge_gauss_r180,
    ge_drag_r180,
    element="qubit",
    rotations="all",
    make_r0=True,
    override=True,
    persist=True,
    d_lambda_map=GE_D_LAMBDA_MAP,
    d_alpha_map=GE_D_ALPHA_MAP,
    d_omega_map=GE_D_OMEGA_MAP,
)

print(f"Registered g→e DRAG-Gaussian rotations: {list(ge_reg_result.keys()) if isinstance(ge_reg_result, dict) else ge_reg_result}")
print(f"  r180 amp: {GE_R180_AMP:.6f}")
print(f"  rlen: {GE_RLEN} ns, sigma: {GE_SIGMA:.3f} ns")
print(f"  DRAG coeff: {GE_DRAG_COEFF}")

## 5. Execution — DRAG-Gaussian Waveforms (e→f Transition)

Build DRAG-corrected Gaussian waveforms for the excited → second-excited (e→f) transition. Register with prefix `ef_` (e.g., `ef_x90`, `ef_x180`). The transition frequency is shifted from the g→e frequency by the anharmonicity.

In [ ]:
ef_gauss_r180, ef_drag_r180 = drag_gaussian_pulse_waveforms(
    EF_R180_AMP, EF_RLEN, EF_SIGMA, EF_DRAG_COEFF, float(attr.anharmonicity)
)

ef_reg_result = register_rotations_from_ref_iq(
    pom,
    ef_gauss_r180,
    ef_drag_r180,
    element="qubit",
    rotations="all",
    make_r0=True,
    override=True,
    persist=True,
    d_lambda_map=EF_D_LAMBDA_MAP,
    d_alpha_map=EF_D_ALPHA_MAP,
    d_omega_map=EF_D_OMEGA_MAP,
    prefix="ef_",
)

print(f"Registered e→f DRAG-Gaussian rotations: {list(ef_reg_result.keys()) if isinstance(ef_reg_result, dict) else ef_reg_result}")
print(f"  r180 amp: {EF_R180_AMP:.6f}")
print(f"  rlen: {EF_RLEN} ns, sigma: {EF_SIGMA:.3f} ns")
print(f"  DRAG coeff: {EF_DRAG_COEFF}")

## 6. Execution — Displacement Operations

Register coherent-state displacement pulses for the storage cavity (e.g., `const_alpha`). These enable the creation of displaced Fock and coherent states in cavity experiments.

In [ ]:
disp_result = ensure_displacement_ops(session)

print(f"Displacement operations registered: {disp_result}")

## 7. Analysis — Waveform Validation Plots

Visualize all defined waveform I/Q envelopes side by side. Verify that amplitudes are within the [−1, 1] DAC range, Gaussian shapes are smooth, and the DRAG correction appears as an imaginary-component feature.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Constant pulse
axes[0, 0].plot(const_I_x180, label="I", color="#2f4b7c")
axes[0, 0].plot(const_Q_x180, label="Q", color="#bc5090")
axes[0, 0].set_title("Constant x180")
axes[0, 0].set_xlabel("Sample")
axes[0, 0].set_ylabel("Amplitude")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# g→e DRAG Gaussian I
axes[0, 1].plot(ge_gauss_r180, label="I (Gaussian)", color="#2f4b7c")
axes[0, 1].set_title("g→e DRAG x180 — I")
axes[0, 1].set_xlabel("Sample")
axes[0, 1].set_ylabel("Amplitude")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# g→e DRAG Gaussian Q
axes[0, 2].plot(ge_drag_r180, label="Q (DRAG)", color="#bc5090")
axes[0, 2].set_title("g→e DRAG x180 — Q")
axes[0, 2].set_xlabel("Sample")
axes[0, 2].set_ylabel("Amplitude")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# e→f DRAG Gaussian I
axes[1, 0].plot(ef_gauss_r180, label="I (Gaussian)", color="#2f4b7c")
axes[1, 0].set_title("e→f DRAG x180 — I")
axes[1, 0].set_xlabel("Sample")
axes[1, 0].set_ylabel("Amplitude")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# e→f DRAG Gaussian Q
axes[1, 1].plot(ef_drag_r180, label="Q (DRAG)", color="#bc5090")
axes[1, 1].set_title("e→f DRAG x180 — Q")
axes[1, 1].set_xlabel("Sample")
axes[1, 1].set_ylabel("Amplitude")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Peak amplitudes comparison
labels = ["const", "g→e", "e→f"]
peaks = [CONST_R180_AMP, GE_R180_AMP, EF_R180_AMP]
colors = ["#4c78a8", "#f58518", "#e45756"]
axes[1, 2].bar(labels, peaks, color=colors)
axes[1, 2].set_title("Peak x180 Amplitudes")
axes[1, 2].set_ylabel("Amplitude")
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Checkpoint — Apply Calibration and Save

Burn the registered pulse operations into the session, persist to disk, and save the stage checkpoint with all waveform parameters.

In [ ]:
session.burn_pulses()
session.save_pulses()

stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="08_pulse_waveform_definition",
    status="calibrated" if APPLY_PULSE_WAVEFORM_CALIBRATION else "characterized",
    summary="Defined constant, g→e DRAG, and e→f DRAG pulse waveforms. Registered rotation families and displacement ops.",
    consumed_inputs={
        "const_r180_amp": CONST_R180_AMP,
        "const_len": CONST_LEN,
        "ge_r180_amp": GE_R180_AMP,
        "ge_rlen": GE_RLEN,
        "ge_sigma": GE_SIGMA,
        "ge_drag_coeff": GE_DRAG_COEFF,
        "ef_r180_amp": EF_R180_AMP,
        "ef_rlen": EF_RLEN,
        "ef_sigma": EF_SIGMA,
        "ef_drag_coeff": EF_DRAG_COEFF,
    },
    persisted_outputs={
        "applied": APPLY_PULSE_WAVEFORM_CALIBRATION,
        "const_rotations": list(const_reg_result.keys()) if isinstance(const_reg_result, dict) else str(const_reg_result),
        "ge_rotations": list(ge_reg_result.keys()) if isinstance(ge_reg_result, dict) else str(ge_reg_result),
        "ef_rotations": list(ef_reg_result.keys()) if isinstance(ef_reg_result, dict) else str(ef_reg_result),
    },
    advisory_outputs={
        "anharmonicity_hz": float(attr.anharmonicity),
    },
    next_stage="09_qutrit_spectroscopy_calibration",
    notes=[
        "Stage 09 assumes g→e and e→f rotation families are registered.",
        "Displacement ops (const_alpha) required for storage cavity experiments.",
    ],
    metrics={
        "const_r180_amp": CONST_R180_AMP,
        "ge_r180_amp": GE_R180_AMP,
        "ef_r180_amp": EF_R180_AMP,
        "ge_drag_coeff": GE_DRAG_COEFF,
        "ef_drag_coeff": EF_DRAG_COEFF,
    },
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")